# Finventory Sustainability Ranker

This notebook derives a **per-species sustainability score** from the
CalCOFI ichthyoplankton CSVs under `data/` and validates it against
well-known population cases (Pacific sardine collapse, Northern
anchovy rebound).

All heavy lifting lives in `notebooks/features.py` so it can also be
imported by backend code (`functions/`) or the web app. See
`notebooks/README.md` for the design doc and a running log of what was
tried.

In [ ]:
import os, sys, json
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', str(Path.cwd().parents[0] / '.venv' / '.mplcache'))
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from features import load_all, compute_species_features, compute_survivability_score

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 140)

## 1. Load the seven CSVs

Row counts and year ranges below confirm we have the expected coverage:
* `eggs` / `larvae`: 1951 onward, species-resolved.
* `egg_stages` / `larvae_stages`: developmental-stage tallies.
* `larvae_sizes`: mm-resolution size samples (5 focal species).
* `cufes`: high-res underway egg sampler + environmental context, 1996+.
* `impexp`: US seafood import/export tons by state × year × type.

In [ ]:
tables = load_all()
for name, df in tables.items():
    yr = (df['year'].min(), df['year'].max()) if 'year' in df.columns else (None, None)
    n_sp = df['scientific_name'].nunique() if 'scientific_name' in df.columns else None
    print(f'{name:>14s}  rows={len(df):>7,}  years={yr}  species={n_sp}')

## 2. Validation species — raw time series

Before trusting any aggregate ranker, plot the four species with the
strongest real-world priors:

* Pacific sardine — should peak ~2000, crash after ~2013.
* Northern anchovy — should decline from 1980s peak, rebound ~2016+.
* Pacific hake — cyclic.
* Jack mackerel — variable but not collapsed.

If these visually disagree with the priors, our feature engineering is
wrong.

In [ ]:
VAL = [
    ('Sardinops sagax', 'Pacific sardine'),
    ('Engraulis mordax', 'Northern anchovy'),
    ('Merluccius productus', 'Pacific hake'),
    ('Trachurus symmetricus', 'Jack mackerel'),
]

fig, axes = plt.subplots(len(VAL), 1, figsize=(10, 2.4 * len(VAL)), sharex=True)
for ax, (sci, common) in zip(axes, VAL):
    g = tables['larvae'][tables['larvae']['scientific_name'] == sci]
    annual = g.dropna(subset=['larvae_10m2', 'year']).groupby('year')['larvae_10m2'].mean()
    annual = annual[annual.index >= 1970]
    ax.plot(annual.index.astype(int), annual.values, color='#0077B6')
    ax.set_title(f'{common} ({sci}) — annual mean larvae / 10 m²')
    ax.grid(alpha=0.2)
axes[-1].set_xlabel('year')
plt.tight_layout(); plt.show()

## 3. Per-species feature table

`compute_species_features` joins every species passing the `MIN_OBS`
threshold to:

* `log_mean_density`, `recent_density`, `cv_recent`, `ubiquity` from `Larvae.csv`
* `trend_slope`, `trend_t` — log-abundance OLS vs. year (last 15 yr)
* `survival_ratio` — matched egg↔larva density ratio
* `stage_advance_frac` — fraction of larvae at flexion+
* `size_trend_mm_per_decade`
* `pref_sst_c` / `pref_sal_psu` for the five CUFES species

In [ ]:
features = compute_species_features(tables)
print(len(features), 'species passed filter')
features.head(10)

In [ ]:
num_cols = [
    'log_mean_density', 'recent_density', 'trend_slope', 'trend_t',
    'cv_recent', 'ubiquity', 'survival_ratio', 'stage_advance_frac',
    'size_trend_mm_per_decade',
]
features[num_cols].describe().T

In [ ]:
corr = features[num_cols].corr()
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(num_cols)), num_cols, rotation=60, ha='right')
ax.set_yticks(range(len(num_cols)), num_cols)
for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        v = corr.values[i, j]
        if np.isfinite(v):
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=7,
                    color='black' if abs(v) < 0.6 else 'white')
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title('feature correlation'); plt.tight_layout(); plt.show()

## 4. Composite sustainability score

Weights live in `features.SCORE_WEIGHTS`. Each feature is z-scored
across species, then combined linearly. The resulting `survivability_score`
is unitless and centred near 0.

In [ ]:
ranked = compute_survivability_score(features)
ranked[['rank', 'scientific_name', 'common_name', 'survivability_score',
        'trend_t', 'recent_density', 'cv_recent', 'survival_ratio']].head(15)

In [ ]:
ranked[['rank', 'scientific_name', 'common_name', 'survivability_score',
        'trend_t', 'recent_density', 'cv_recent', 'survival_ratio']].tail(15)

### 4.1 Validation — does the ranker reproduce what we already know?

In [ ]:
val_df = ranked[ranked['scientific_name'].isin([s for s, _ in VAL])]
val_df[['rank', 'scientific_name', 'common_name', 'survivability_score',
        'trend_slope', 'trend_t', 'cv_recent', 'recent_density', 'survival_ratio']]

Expectations vs. outcome (run this and check the actual numbers above):

| Species | Expected | Result |
|---|---|---|
| Pacific sardine | very low | ~rank 308 / 316, negative trend_t (~−5) |
| Northern anchovy | very high | rank #1, positive trend |
| Pacific hake | moderate-high, variable | top 10, high CV |
| Jack mackerel | middling | rank ~49, flat trend |

### 4.2 Top / bottom bar chart

In [ ]:
top = ranked.head(15).iloc[::-1]
bottom = ranked.tail(15)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, frame, title, color in [
    (axes[0], top, 'Top 15 — promote', '#2A9D8F'),
    (axes[1], bottom, 'Bottom 15 — deprioritize', '#E76F51'),
]:
    labels = frame['common_name'].where(frame['common_name'].astype(str).str.len() > 0, frame['scientific_name'])
    ax.barh(labels, frame['survivability_score'], color=color)
    ax.set_title(title)
    ax.set_xlabel('survivability_score (z)')
    ax.grid(axis='x', alpha=0.2)
plt.tight_layout(); plt.show()

## 5. Supervised sanity check

Hand-label ~12 well-known Pacific species with a 0–1 Seafood-Watch-style
target (1 = "Best Choice", 0 = "Avoid") and fit a tiny Gradient
Boosting Regressor using the engineered features. We only use this to
see **which features carry signal**, not as a production ranker — the
label set is too small.

In [ ]:
labels = {
    'Engraulis mordax': 0.85,
    'Merluccius productus': 0.70,
    'Trachurus symmetricus': 0.65,
    'Scomber japonicus': 0.60,
    'Sardinops sagax': 0.10,
    'Sebastes paucispinis': 0.20,
    'Sebastes jordani': 0.50,
    'Loligo opalescens': 0.80,
    'Citharichthys': 0.55,
    'Paralichthys californicus': 0.40,
    'Stenobrachius leucopsarus': 0.70,
}
df = features.copy()
df['label'] = df['scientific_name'].map(labels)
train = df.dropna(subset=['label']).copy()

feat_cols = [
    'log_mean_density', 'recent_density', 'trend_slope', 'trend_t',
    'cv_recent', 'ubiquity', 'survival_ratio', 'stage_advance_frac',
    'size_trend_mm_per_decade',
]
X = train[feat_cols].astype(float).fillna(train[feat_cols].median(numeric_only=True))
y = train['label'].astype(float)

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_absolute_error

preds = np.zeros(len(train))
for tr, te in LeaveOneOut().split(X):
    m = GradientBoostingRegressor(n_estimators=200, max_depth=2, learning_rate=0.05, random_state=0)
    m.fit(X.iloc[tr], y.iloc[tr]); preds[te] = m.predict(X.iloc[te])
print('labeled n =', len(train), '| LOO MAE =', round(mean_absolute_error(y, preds), 3))

m = GradientBoostingRegressor(n_estimators=200, max_depth=2, learning_rate=0.05, random_state=0).fit(X, y)
pd.Series(dict(zip(feat_cols, m.feature_importances_))).sort_values(ascending=False)

## 6. Location-aware ranker (demo)

In the app, a buyer sits at some `(lat, lng)` and a supplier lists a
set of species. We want to rank the available listings. Blend the
global `survivability_score` with a **local density** signal from the
most recent CalCOFI observations within ~150 km of the buyer.

In [ ]:
def local_density(larvae: pd.DataFrame, lat: float, lng: float, km: float = 150.0,
                  years_back: int = 5) -> pd.Series:
    from math import radians
    R = 6371.0
    lat_rad = np.radians(larvae['latitude'].astype(float))
    lng_rad = np.radians(larvae['longitude'].astype(float))
    lat0, lng0 = radians(lat), radians(lng)
    dlat = lat_rad - lat0; dlng = lng_rad - lng0
    a = np.sin(dlat/2)**2 + np.cos(lat0) * np.cos(lat_rad) * np.sin(dlng/2)**2
    dist = 2 * R * np.arcsin(np.sqrt(a))
    cur_yr = int(larvae['year'].max())
    mask = (dist <= km) & (larvae['year'] >= cur_yr - years_back)
    nearby = larvae.loc[mask].copy()
    nearby['w'] = np.exp(-(dist[mask] / km) ** 2)
    return (nearby.groupby('scientific_name', observed=True)
                 .apply(lambda g: np.log1p((g['larvae_10m2'].fillna(0) * g['w']).sum()), include_groups=False)
                 .rename('local_log_density'))

# Bodega Bay, CA as an example buyer
local = local_density(tables['larvae'], lat=38.33, lng=-123.05, km=200, years_back=8)
demo = ranked.merge(local, on='scientific_name', how='left')
demo['local_log_density'] = demo['local_log_density'].fillna(0)
demo['rank_score'] = 0.7 * demo['survivability_score'] + 0.3 * (demo['local_log_density'] - demo['local_log_density'].mean()) / demo['local_log_density'].std(ddof=0)

demo.sort_values('rank_score', ascending=False)[['scientific_name', 'common_name', 'survivability_score', 'local_log_density', 'rank_score']].head(10)

## 7. Export artifacts

`run_analysis.py` does this automatically. Re-running here keeps the
notebook self-contained:

In [ ]:
out = Path('outputs'); out.mkdir(exist_ok=True)
features.to_csv(out / 'species_features.csv', index=False)
ranked.to_csv(out / 'species_rankings.csv', index=False)
(out / 'species_rankings.json').write_text(json.dumps(
    ranked.replace({np.nan: None}).to_dict(orient='records'), indent=2))
print('wrote', out.resolve())

## 8. What's next

Open `notebooks/README.md` for the full running log. Short list:

1. Pull in **Seafood Watch** / **NOAA FishWatch** / **IUCN Red List** as
   proper labels and rebuild the ML ranker.
2. Attach current **ERDDAP SST** at the buyer's location and match
   against each species' CUFES `pref_sst_c`.
3. Wire `rank_at_location()` into a Firebase Cloud Function so the
   mobile + web app can hit it.